# Fink/LSST — Reload, Renormalise and Display Saved Light Curves (Tags × DDFs)

This notebook reads the Parquet files saved by `07_fink_tags_lightcurves.ipynb`  
and extends notebook `08_fink_tags_lightcurves_replot.ipynb` by adding  
**inter-band flux renormalisation** inspired by `02_fink_block_lightcurves_replot_renormalised.ipynb`.

**Expected data** in `data_FINK_BLOCK_LC_07/`:
- `{tag}_fp.parquet`  — forced-photometry light curves
- `{tag}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics
- `tag_ranking.csv`   — variability ranking by tag
- `visit_summary_src.csv` / `visit_summary_fp.csv` — visit-level summaries

No API call is made in this notebook.

## Key differences vs notebook 08 (replot without renormalisation)

| Aspect | NB 08 (replot) | NB 08-renorm (this) |
|--------|---------------|--------------------|
| `lc_cache` structure | `lc_cache[oid]` | identical |
| Renormalisation | absent | added (sections 11–13) |
| fp sanitisation | yes (safe helpers) | yes — preserved |
| New-alert handling | empty-fp guard | same guards |

## Key differences vs notebook 02 (block renormalisation)

| Aspect | NB 02 (blocks) | NB 08-renorm (this) |
|--------|---------------|--------------------|
| `lc_cache` structure | `lc_cache[group][oid]` | `lc_cache[oid]` with `group`/`tag`/`field` keys |
| `rank_oids` signature | `rank_oids(group)` | `rank_oids(oid_list)` |
| `plot_singleobjlightcurve` | accesses `lc_cache[group][oid]` | accesses `lc_cache[oid]` |
| fp data availability | always present | may be empty (new alerts) → sanitised |


- **author** : Sylvie Dagoret-Campane
- **affiliation** : IJCLab/CNRS
- **last update** : 2026-03-28
- **subject:** Fink alert Broker applied to Rubin LSST alerts — tags renormalisation

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import collections
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 07
DIR_FIGS = f"figs_{NB_TAG}_08renorm"  # output: figures for this renorm notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per tag
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ─────────────────────────────────────────────────────────────
# Choose which tags to display:
#   'all'  -> all tags found on disk (default)
#   list   -> subset of tags, e.g. ['in_tns', 'hostless_candidate']
PLOT_MODE = "all"  # <── change here: 'all' | list of tags

# ── Known Fink tag descriptions (used for display) ───────────────────────────
FINK_TAGS = {
    "extragalactic_lt20mag_candidate": "Rising, bright (mag < 20), extragalactic candidates",
    "extragalactic_new_candidate": "New (< 48 h first detection) and potentially extragalactic",
    "hostless_candidate": "Hostless alerts according to ELEPHANT (arXiv:2404.18165)",
    "in_tns": "Alerts with a known counterpart in TNS (AT or confirmed)",
    "sn_near_galaxy_candidate": "Alerts matching a galaxy catalog and consistent with SNe",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (+ luptitude support)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes).

    Luptitudes (Lupton et al. 1999) handle zero and negative flux values
    that arise in difference-image photometry (DIA).  They behave as
    standard AB magnitudes at high signal-to-noise and transition to a
    linear flux scale near the noise floor, so every measurement is
    plotted -- including those with negative flux.

    The softening parameter *b* is set to the median flux uncertainty of
    the input array, placing the linear/log transition at the 1-sigma
    noise level.

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.  May contain negative values.
    flux_err_nJy : array-like
        1-sigma flux uncertainty in nJy.
    zero_point : float, optional
        AB zero-point offset in nJy units (default 31.4 for nJy).

    Returns
    -------
    lup : ndarray
    lup_err : ndarray
    b : float — softening parameter used
    """
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)

    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0

    log10_e_inv = 1.085736

    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)

    return lup, lup_err, b


print("flux_to_luptitude() defined.")

## 2b. Safe helpers for new-alert objects (fp may be empty)

Some Fink tags (e.g. `extragalactic_new_candidate`) contain alerts that were
detected for the first time recently and therefore have **no forced-photometry
(fp) data**.  The corresponding DataFrames are empty but retain their column
schema.  The helpers below handle this case gracefully without raising
KeyError or ValueError.

In [ ]:
def safe_filter_band(df, band, band_col="r:band", time_col="r:midpointMjdTai"):
    """Return rows where band_col == band, handling empty or schema-only DataFrames."""
    if df is None or df.empty:
        return pd.DataFrame(columns=[band_col, time_col])
    if band_col not in df.columns:
        return pd.DataFrame(columns=df.columns)
    df_b = df[df[band_col] == band]
    if time_col in df_b.columns:
        df_b = df_b.sort_values(time_col)
    return df_b


def safe_finite_mask(df, cols):
    """Return a boolean mask of rows where all *cols* are finite."""
    if df is None or df.empty:
        return np.array([], dtype=bool)
    for c in cols:
        if c not in df.columns:
            return np.zeros(len(df), dtype=bool)
    mask = np.ones(len(df), dtype=bool)
    for c in cols:
        mask &= np.isfinite(df[c].to_numpy())
    return mask


def safe_get_array(df, col):
    """Return df[col].to_numpy(), or empty array if df is empty or col absent."""
    if df is None or df.empty or col not in df.columns:
        return np.array([])
    return df[col].to_numpy()


print("Safe helpers defined (safe_filter_band / safe_finite_mask / safe_get_array).")

## 3. Discover available tags from Parquet files on disk

In [ ]:
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def tag_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


tags_fp = {tag_from_path(p, "_fp.parquet"): p for p in fp_files}
tags_src = {tag_from_path(p, "_src.parquet"): p for p in src_files}
all_tags = sorted(set(tags_fp) | set(tags_src))

if PLOT_MODE == "all":
    selected_tags = list(all_tags)
elif isinstance(PLOT_MODE, list):
    selected_tags = [t for t in all_tags if t in PLOT_MODE]
else:
    selected_tags = list(all_tags)
    print(f"WARNING: unknown PLOT_MODE={PLOT_MODE!r}, defaulting to 'all'")

print(f"Tags on disk: {len(all_tags)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_tags)}")
print()
for t in all_tags:
    has_fp = "yes" if t in tags_fp else "no "
    has_src = "yes" if t in tags_src else "no "
    sel = "<<" if t in selected_tags else "  "
    desc = FINK_TAGS.get(t, "(unknown tag)")
    print(f"  {sel} {t:50s}  fp={has_fp}  src={has_src}  |  {desc}")

## 4. Load Parquet files — reconstruct `lc_cache`

The cache structure mirrors notebook 07:
```python
lc_cache[oid] = {
    'fp'   : DataFrame,   # forced photometry  (may be empty for new alerts)
    'src'  : DataFrame,   # diaSources
    'group': str,         # = tag name
    'tag'  : str,         # = tag name
    'field': str,         # DDF name
}
```

**New-alert sanitisation**: when a tag's `_fp.parquet` file does not exist or is
empty, the `'fp'` entry is initialised to an empty DataFrame with the expected
column schema so that all downstream code can assume the key exists.

In [ ]:
# Minimal column schema expected for fp DataFrames.
# Used to build an empty-but-valid placeholder when fp data is absent.
_FP_SCHEMA_COLS = [
    "r:midpointMjdTai",
    "r:psfFlux",
    "r:psfFluxErr",
    "r:band",
    "diaObjectId",
    "group",
    "tag",
    "field",
    "mag",
    "mag_err",
]


def _empty_fp_df():
    """Return an empty DataFrame with the expected fp column schema."""
    return pd.DataFrame(columns=_FP_SCHEMA_COLS)


lc_cache = {}  # oid (str) → {fp, src, group, tag, field}

for tag in all_tags:
    for lc_type, path_dict in (("fp", tags_fp), ("src", tags_src)):
        if tag not in path_dict:
            continue
        df = pd.read_parquet(path_dict[tag])

        # Drop residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag/mag_err if absent
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        oid_col = "diaObjectId" if "diaObjectId" in df.columns else "r:diaObjectId"

        for oid, sub in df.groupby(oid_col):
            oid = str(oid)
            if oid not in lc_cache:
                row0 = sub.iloc[0]
                lc_cache[oid] = {
                    "fp": _empty_fp_df(),  # always initialised — sanitised for new alerts
                    "src": pd.DataFrame(),
                    "group": str(row0.get("group", tag)),
                    "tag": str(row0.get("tag", tag)),
                    "field": str(row0.get("field", "unknown")),
                }
            lc_cache[oid][lc_type] = sub.reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total objects loaded: {len(lc_cache)}")
print()

all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Objects loaded per tag   [DDF breakdown]:")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    fields_in_group = collections.Counter(lc_cache[o]["field"] for o in group_oids[g])
    field_str = "  ".join(f"{f}:{n}" for f, n in sorted(fields_in_group.items()))
    n_pts = sum(len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"]) for o in group_oids[g])
    fp_empty = sum(1 for o in group_oids[g] if lc_cache[o]["fp"].empty)
    print(f"  {g:50s}  {len(group_oids[g]):3d} objects  {n_pts:6d} pts   fp_empty={fp_empty}   {field_str}")

## 5. Load flatness metrics and tag ranking

In [ ]:
csv_path = os.path.join(DIR_DATA, "flatness_metrics.csv")
if os.path.exists(csv_path):
    df_flat = pd.read_csv(csv_path)
    print(f"flatness_metrics.csv loaded: {len(df_flat)} rows")
    print("\nMedian flatness by tag:")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
    print("\nMedian flatness by DDF:")
    if "field" in df_flat.columns:
        print(df_flat.groupby("field")[["n_pts", "rms_var"]].median().round(4))
else:
    df_flat = pd.DataFrame()
    print("flatness_metrics.csv not found — flatness plots will be skipped.")

ranking_path = os.path.join(DIR_DATA, "tag_ranking.csv")
if os.path.exists(ranking_path):
    df_ranking = pd.read_csv(ranking_path)
    print(f"\ntag_ranking.csv loaded: {len(df_ranking)} rows")
else:
    df_ranking = pd.DataFrame()
    print("\ntag_ranking.csv not found.")

## 6. Flatness boxplot per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle(
        "Flux variability by Fink tag (all DDFs combined) — lower = flatter",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_tag")
    plt.show()

## 6b. Flatness boxplot per DDF (all tags combined)

In [ ]:
if df_flat.empty or "field" not in df_flat.columns:
    print("No flatness / field data available.")
else:
    fields_present = sorted(df_flat["field"].dropna().unique())
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_field = [df_b[df_b["field"] == f]["rms_var"].dropna().values for f in fields_present]
        bp = ax.boxplot(
            data_per_field, labels=fields_present, patch_artist=True, notch=False, showfliers=True
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by DDF (all tags combined)", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01b_flatness_boxplot_by_field")
    plt.show()

## 7. Common helpers for light-curve grids

In [ ]:
def rank_oids(oid_list, nc=NC):
    """Return the top-nc object IDs, ranked by total number of data points."""
    scored = [(o, len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def getTminTmax(df, df_src, df_fp):
    """Compute [tmin, tmax] over all available time arrays.

    tmin is anchored to the first *src* detection when src is available.
    Handles empty DataFrames gracefully.
    """
    t = df["r:midpointMjdTai"].to_numpy() if "r:midpointMjdTai" in df.columns else np.array([])
    t_src = df_src["r:midpointMjdTai"].to_numpy() if "r:midpointMjdTai" in df_src.columns else np.array([])
    t_fp = df_fp["r:midpointMjdTai"].to_numpy() if "r:midpointMjdTai" in df_fp.columns else np.array([])

    if len(t_src) > 0:
        tmin = t_src.min()
    elif len(t) > 0:
        tmin = t.min()
    else:
        tmin = 0.0

    tmax = t.max() if len(t) > 0 else tmin
    return tmin, tmax


def clean_df(df):
    """Deduplicate, drop NaN core columns, sort by time."""
    if df is None or df.empty:
        return df if df is not None else pd.DataFrame()
    return (
        df.drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
        .sort_values("r:midpointMjdTai")
        .reset_index(drop=True)
    )


print("Common helpers defined (rank_oids / getTminTmax / clean_df).")

## 8. Grid light-curve plotting function (without renormalisation)

Identical to notebook `08_fink_tags_lightcurves_replot.ipynb`.  Used for
sections 9, 9b, and 10 (raw flux / mag / luptitude) before the renormalised
views in sections 11–13.

In [ ]:
def plot_lc_grid(oid_list, group, mode="flux", nc=NC):
    """
    Plot a (n_objects × n_bands) grid of light curves for one Fink tag.

    Parameters
    ----------
    oid_list : list of str
    group    : str — tag name (used for title and file name)
    mode     : 'flux' | 'mag' | 'luptitude'
    nc       : int — maximum number of objects to plot
    """

    STYLE = {
        "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),
        "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),
    }

    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for tag {group}.")
        return

    fig, axes = plt.subplots(
        n_obj,
        len(BANDS) + 1,
        figsize=(2.8 * (len(BANDS) + 1), 2.6 * n_obj),
        sharex=False,
        sharey=False,
        squeeze=False,
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        field = d.get("field", "")

        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        df_all = clean_df(
            pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        )
        df_src = clean_df(d["src"])
        df_fp = clean_df(d["fp"])

        tmin, tmax = getTminTmax(df_all, df_src, df_fp)
        t_all = df_all["r:midpointMjdTai"].values
        dtmin = t_all.min() - tmin if len(t_all) > 0 else 0
        dtmax = tmax - tmin

        ax_first_band = None
        ax_last_allbands = axes[row_idx][len(BANDS)]

        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = safe_filter_band(df_all, band)
            df_b_src = safe_filter_band(df_src, band)
            df_b_fp = safe_filter_band(df_fp, band)

            if len(df_b) < 3:
                ax.set_visible(False)
                continue
            if ax_first_band is None:
                ax_first_band = ax

            # ── y-values ────────────────────────────────────────────────
            if mode == "flux":
                mask = safe_finite_mask(df_b, ["r:psfFlux", "r:psfFluxErr"])
                mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
                mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
                df_b = df_b[mask].reset_index(drop=True)
                df_b_src = df_b_src[mask_src].reset_index(drop=True)
                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue
                y_src = safe_get_array(df_b_src, "r:psfFlux")
                yerr_src = safe_get_array(df_b_src, "r:psfFluxErr")
                y_fp = safe_get_array(df_b_fp[mask_fp] if len(mask_fp) > 0 else df_b_fp, "r:psfFlux")
                yerr_fp = safe_get_array(df_b_fp[mask_fp] if len(mask_fp) > 0 else df_b_fp, "r:psfFluxErr")

            elif mode == "mag":
                mask = safe_finite_mask(df_b, ["mag", "mag_err"])
                mask_src = safe_finite_mask(df_b_src, ["mag", "mag_err"])
                mask_fp = safe_finite_mask(df_b_fp, ["mag", "mag_err"])
                df_b = df_b[mask].reset_index(drop=True)
                df_b_src = df_b_src[mask_src].reset_index(drop=True)
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True) if len(mask_fp) > 0 else df_b_fp
                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue
                y_src = safe_get_array(df_b_src, "mag")
                yerr_src = safe_get_array(df_b_src, "mag_err")
                y_fp = safe_get_array(df_b_fp, "mag")
                yerr_fp = safe_get_array(df_b_fp, "mag_err")
                ax.invert_yaxis()

            elif mode == "luptitude":
                mask = safe_finite_mask(df_b, ["r:psfFlux", "r:psfFluxErr"])
                mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
                mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
                df_b = df_b[mask].reset_index(drop=True)
                df_b_src = df_b_src[mask_src].reset_index(drop=True)
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True) if len(mask_fp) > 0 else df_b_fp
                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue
                flux_src = safe_get_array(df_b_src, "r:psfFlux")
                fluxerr_src = safe_get_array(df_b_src, "r:psfFluxErr")
                flux_fp = safe_get_array(df_b_fp, "r:psfFlux")
                fluxerr_fp = safe_get_array(df_b_fp, "r:psfFluxErr")
                y_src, yerr_src, b_soft = flux_to_luptitude(flux_src, fluxerr_src)
                y_fp, yerr_fp, _ = flux_to_luptitude(flux_fp, fluxerr_fp)
                ax.invert_yaxis()
            else:
                raise ValueError(f"Unknown mode {mode!r}.")

            # ── x-values ────────────────────────────────────────────────
            dt_src = safe_get_array(df_b_src, "r:midpointMjdTai") - tmin
            dt_fp = safe_get_array(
                df_b_fp if len(safe_get_array(df_b_fp, "r:midpointMjdTai")) > 0 else pd.DataFrame(),
                "r:midpointMjdTai",
            )
            if len(dt_fp) > 0:
                dt_fp = dt_fp - tmin

            color = BAND_COLORS.get(band, "gray")

            ax.errorbar(dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"])
            if len(y_fp) > 0:
                ax.errorbar(dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"])
            ax_last_allbands.errorbar(
                dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"]
            )
            if len(y_fp) > 0:
                ax_last_allbands.errorbar(
                    dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"]
                )

            rms = rms_variability(df_b["r:psfFlux"].values)
            if mode == "luptitude":
                ax.set_title(f"{band} N={len(df_b)} b={b_soft:.1f}nJy", fontsize=7, pad=2, color=color)
            else:
                ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)
            ax.set_xlim(dtmin - 2, dtmax + 2)

        _ylabel_map = {"flux": "flux (nJy)", "mag": "AB mag", "luptitude": "asinh mag (luptitude)"}
        ylabel_unit = _ylabel_map.get(mode, mode)

        if ax_first_band is not None:
            deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
            field_label = f"  [{deep_field}]" if deep_field else ""
            ax_first_band.set_ylabel(f"{oid}{field_label}\n{ylabel_unit}", fontsize=10)
        else:
            axes[row_idx][0].set_ylabel(f"{oid}\n{ylabel_unit}", fontsize=8)

        deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
        ax_last_allbands.set_xlabel("Δt (days)", fontsize=7)
        ax_last_allbands.set_xlim(dtmin - 2, dtmax + 2)
        ax_last_allbands.tick_params(labelsize=7)
        ax_last_allbands.set_ylabel(f"{ylabel_unit}", fontsize=10)
        if mode in ("mag", "luptitude"):
            ax_last_allbands.invert_yaxis()
        ax_last_allbands.set_title(f"{oid} ({deep_field})", fontsize=8)

    fig.suptitle(f"Tag: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("plot_lc_grid() defined.")

## 9. Light curves in flux (nJy) — raw

In [ ]:
groups_to_plot = [g for g in selected_tags if g in group_oids and len(group_oids[g]) >= 1]

for group in groups_to_plot:
    n_obj = len(group_oids[group])
    desc = FINK_TAGS.get(group, "")
    print(f"\n=== {group} ({n_obj} objects) — flux ===")
    if desc:
        print(f"    {desc}")
    plot_lc_grid(group_oids[group], group, mode="flux")

## 9b. Light curves as Luptitudes (asinh magnitudes) — raw

In [ ]:
for group in groups_to_plot:
    n_obj = len(group_oids[group])
    desc = FINK_TAGS.get(group, "")
    print(f"\n=== {group} ({n_obj} objects) — luptitude ===")
    if desc:
        print(f"    {desc}")
    plot_lc_grid(group_oids[group], group, mode="luptitude")

## 10. Detailed view — single object inspector (flux + magnitude + luptitude)

Set `TARGET_TAG` and `TARGET_OID` below to inspect any individual object.

In [ ]:
TARGET_TAG = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(group_oids[TARGET_TAG], nc=20)
TARGET_OID = TOP_OIDS[0]  # ← change here

print(f"Tag      : {TARGET_TAG}")
print(f"Object   : {TARGET_OID}")
print(f"DDF      : {lc_cache[TARGET_OID]['field']}")
print(f"Top OIDs : {TOP_OIDS}")

In [ ]:
def plot_singleobjlightcurve(
    lc_cache,
    target_group,
    target_oid,
    bands,
    band_colors,
    modes=("flux", "mag", "luptitude"),
    show_per_band=True,
    show_combined=True,
    style=None,
    save=False,
    filename_prefix="03_detail",
):
    """Detailed single-object light curve viewer (no renormalisation).

    Adapted from NB 08 to use the flat lc_cache[oid] structure and
    safe helpers for objects with empty fp DataFrames.
    """
    if style is None:
        style = {
            "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),
            "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),
        }

    d = lc_cache[target_oid]

    frames = [df for df in (d["fp"], d["src"]) if not df.empty]
    df_obj = clean_df(
        pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    )
    df_obj_src = clean_df(d["src"])
    df_obj_fp = clean_df(d["fp"])

    tmin, tmax = getTminTmax(df_obj, df_obj_src, df_obj_fp)
    bands_obj = [b for b in bands if b in df_obj["r:band"].values]

    n_rows = (len(bands_obj) if show_per_band else 0) + (1 if show_combined else 0)
    n_cols = len(modes)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.2 * n_rows), squeeze=False)
    mode_to_col = {m: i for i, m in enumerate(modes)}
    current_row = 0

    def plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=False):
        t_src = safe_get_array(df_b_src, "r:midpointMjdTai")
        dt_src = t_src - tmin
        t_fp = safe_get_array(df_b_fp, "r:midpointMjdTai")
        dt_fp = t_fp - tmin

        if mode == "flux":
            mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
            mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
            ax.errorbar(
                dt_src[mask_src],
                safe_get_array(df_b_src, "r:psfFlux")[mask_src],
                yerr=safe_get_array(df_b_src, "r:psfFluxErr")[mask_src],
                color=color,
                **style["src"],
            )
            if mask_fp.sum() > 0:
                ax.errorbar(
                    dt_fp[mask_fp],
                    safe_get_array(df_b_fp, "r:psfFlux")[mask_fp],
                    yerr=safe_get_array(df_b_fp, "r:psfFluxErr")[mask_fp],
                    color=color,
                    **style["fp"],
                )
            ax.axhline(0, color="black", lw=0.6, ls="--", alpha=0.4)
            ax.set_ylabel("Flux (nJy)")

        elif mode == "mag":
            mask_src = safe_finite_mask(df_b_src, ["mag", "mag_err"])
            mask_fp = safe_finite_mask(df_b_fp, ["mag", "mag_err"])
            ax.errorbar(
                dt_src[mask_src],
                safe_get_array(df_b_src, "mag")[mask_src],
                yerr=safe_get_array(df_b_src, "mag_err")[mask_src],
                color=color,
                **style["src"],
            )
            if mask_fp.sum() > 0:
                ax.errorbar(
                    dt_fp[mask_fp],
                    safe_get_array(df_b_fp, "mag")[mask_fp],
                    yerr=safe_get_array(df_b_fp, "mag_err")[mask_fp],
                    color=color,
                    **style["fp"],
                )
            ax.invert_yaxis()
            ax.set_ylabel("AB mag")

        elif mode == "luptitude":
            mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
            mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
            if mask_src.sum() >= 3:
                lup_src, lup_err_src, _ = flux_to_luptitude(
                    safe_get_array(df_b_src, "r:psfFlux")[mask_src],
                    safe_get_array(df_b_src, "r:psfFluxErr")[mask_src],
                )
                ax.errorbar(dt_src[mask_src], lup_src, yerr=lup_err_src, color=color, **style["src"])
                if mask_fp.sum() > 0:
                    lup_fp, lup_err_fp, _ = flux_to_luptitude(
                        safe_get_array(df_b_fp, "r:psfFlux")[mask_fp],
                        safe_get_array(df_b_fp, "r:psfFluxErr")[mask_fp],
                    )
                    ax.errorbar(dt_fp[mask_fp], lup_fp, yerr=lup_err_fp, color=color, **style["fp"])
                ax.invert_yaxis()
                ax.set_ylabel("Luptitude")

        ax.set_xlabel("Δt (days)")
        if not is_combined:
            ax.set_title(f"{mode} — band {band}", fontsize=8, color=color)

    if show_per_band:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = safe_filter_band(df_obj_src, band)
            df_b_fp = safe_filter_band(df_obj_fp, band)
            color = band_colors.get(band, "gray")
            for mode in modes:
                plot_one(axes[current_row][mode_to_col[mode]], df_b, df_b_src, df_b_fp, mode, band, color)
            current_row += 1

    if show_combined:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = safe_filter_band(df_obj_src, band)
            df_b_fp = safe_filter_band(df_obj_fp, band)
            color = band_colors.get(band, "gray")
            for mode in modes:
                plot_one(
                    axes[current_row][mode_to_col[mode]],
                    df_b,
                    df_b_src,
                    df_b_fp,
                    mode,
                    band,
                    color,
                    is_combined=True,
                )
        for mode in modes:
            ax = axes[current_row][mode_to_col[mode]]
            ax.set_title(f"{mode} (all bands)")
            if mode in ("mag", "luptitude"):
                ax.invert_yaxis()

    fig.suptitle(f"diaObjectId = {target_oid} | group = {target_group}", fontsize=11)
    plt.tight_layout()
    if save:
        plt.savefig(f"{filename_prefix}_{target_oid}.png")
    plt.show()


print("plot_singleobjlightcurve() defined.")

In [ ]:
plot_singleobjlightcurve(
    lc_cache,
    TARGET_TAG,
    TARGET_OID,
    BANDS,
    BAND_COLORS,
    modes=("flux", "mag", "luptitude"),
    show_per_band=True,
    show_combined=True,
)

## 11. Inter-band renormalisation — helper functions

The strategy is identical to notebook `02_fink_block_lightcurves_replot_renormalised.ipynb`
but adapted to the flat `lc_cache[oid]` structure used by the tags notebooks.

### Principle

For each object, a reference band is chosen (priority: *i → r → z → g → y*)
and the other bands are scaled / offset to match it:

- **Flux** : multiplicative factor = `median(f_ref / f_b)` after temporal interpolation.
- **Magnitude / luptitude** : additive offset = `median(m_ref - m_b)`.

### Special handling for new-alert objects

Objects with empty fp DataFrames are handled transparently because
`apply_normalization` works on copies and `safe_filter_band` returns an
empty DataFrame when the source is empty.

In [ ]:
def choose_reference_band(df_src, df_all, bands_priority=("i", "r", "z", "g", "y"), min_pts=5):
    """Choose the best reference band for inter-band renormalisation.

    Priority order: i → r → z → g → y (central bands first).
    Falls back to the combined src+fp DataFrame if src alone has too few points.

    Returns (band, source) or (None, None) when no band qualifies.
    """
    for b in bands_priority:
        if df_src is not None and not df_src.empty:
            if (df_src["r:band"] == b).sum() >= min_pts:
                return b, "src"
    for b in bands_priority:
        if df_all is not None and not df_all.empty:
            if (df_all["r:band"] == b).sum() >= min_pts:
                return b, "all"
    return None, None


def select_norm_dataset(df_src, df_all, band, min_pts=5):
    """Return the dataset (src or src+fp) to use for normalising *band*."""
    if df_src is not None and not df_src.empty:
        dfb = df_src[df_src["r:band"] == band]
        if len(dfb) >= min_pts:
            return dfb, "src"
    if df_all is not None and not df_all.empty:
        dfb = df_all[df_all["r:band"] == band]
        if len(dfb) >= min_pts:
            return dfb, "all"
    return None, None


def compute_band_normalization(df_ref, df_b, quantity="flux", min_overlap=5):
    """Estimate the normalisation factor/offset between *df_ref* and *df_b*.

    Temporal interpolation is used to align the two time series before
    computing the median ratio (flux) or median difference (mag/luptitude).

    Parameters
    ----------
    df_ref, df_b : DataFrame — must contain 'r:midpointMjdTai' and the
        appropriate flux/mag columns.
    quantity : 'flux' | 'mag' | 'luptitude'
    min_overlap : int — minimum number of overlapping points required.

    Returns
    -------
    float (factor or offset) or None if insufficient overlap.
    """
    if df_ref is None or df_b is None or df_ref.empty or df_b.empty:
        return None

    df_ref = df_ref.sort_values("r:midpointMjdTai")
    df_b = df_b.sort_values("r:midpointMjdTai")

    t_ref = df_ref["r:midpointMjdTai"].values
    t_b = df_b["r:midpointMjdTai"].values

    if quantity == "flux":
        y_ref = df_ref["r:psfFlux"].values
        y_b = df_b["r:psfFlux"].values
    elif quantity == "mag":
        if "mag" not in df_ref.columns or "mag" not in df_b.columns:
            return None
        y_ref = df_ref["mag"].values
        y_b = df_b["mag"].values
    elif quantity == "luptitude":
        y_ref, _, _ = flux_to_luptitude(df_ref["r:psfFlux"].values, df_ref["r:psfFluxErr"].values)
        y_b, _, _ = flux_to_luptitude(df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values)
    else:
        raise ValueError(f"Unknown quantity {quantity!r}.")

    y_b_interp = np.interp(t_ref, t_b, y_b, left=np.nan, right=np.nan)
    mask = np.isfinite(y_ref) & np.isfinite(y_b_interp)

    if mask.sum() < min_overlap:
        return None

    if quantity == "flux":
        # Avoid division by zero
        valid = mask & (np.abs(y_b_interp) > 0)
        if valid.sum() < min_overlap:
            return None
        return float(np.nanmedian(y_ref[valid] / y_b_interp[valid]))
    else:
        return float(np.nanmedian(y_ref[mask] - y_b_interp[mask]))


def apply_normalization(df, band, factor, quantity="flux"):
    """Apply a renormalisation factor/offset to *band* rows in *df*.

    Returns a copy; the original DataFrame is never modified.
    """
    if df is None or df.empty or factor is None:
        return df

    df = df.copy()
    mask = df["r:band"] == band

    if quantity == "flux":
        df.loc[mask, "r:psfFlux"] = df.loc[mask, "r:psfFlux"] * factor
        df.loc[mask, "r:psfFluxErr"] = df.loc[mask, "r:psfFluxErr"] * factor
        # Recompute magnitude columns after flux rescaling
        flux_vals = df.loc[mask, "r:psfFlux"].values
        err_vals = df.loc[mask, "r:psfFluxErr"].values
        mag_vals, mag_err_vals = flux_to_mag(flux_vals, err_vals)
        df.loc[mask, "mag"] = mag_vals
        df.loc[mask, "mag_err"] = mag_err_vals

    elif quantity == "mag":
        df.loc[mask, "mag"] = df.loc[mask, "mag"] + factor

    # For 'luptitude': renormalise through flux only (luptitudes are recomputed
    # on the fly in plot_one; they are not stored in the DataFrame).

    return df


print("Renormalisation helpers defined.")

## 12. Renormalised single-object viewer

Extends `plot_singleobjlightcurve()` with inter-band renormalisation.
The normalisation is computed on working copies of the DataFrames and does
**not** modify `lc_cache`.

In [ ]:
def plot_singleobjlightcurve_normalized(
    lc_cache,
    target_group,
    target_oid,
    bands,
    band_colors,
    modes=("flux", "mag", "luptitude"),
    show_per_band=True,
    show_combined=True,
    normalize=True,
    norm_quantity="flux",  # 'flux' | 'mag' | 'luptitude'
    min_pts=5,
    min_overlap=5,
    style=None,
    save=False,
    filename_prefix="04_detail_normalized",
):
    """Single-object light curve viewer with optional inter-band renormalisation.

    Adapted from NB 02 (`plot_singleobjlightcurve_normalized`) to use the flat
    `lc_cache[oid]` structure of the tags notebooks and safe helpers for objects
    with empty fp DataFrames (new alerts).

    Parameters
    ----------
    normalize    : bool — apply inter-band renormalisation.
    norm_quantity: 'flux' | 'mag' | 'luptitude'
        'luptitude' normalises via flux (luptitudes are recomputed on the fly).
    min_pts      : minimum points required to qualify a band as reference.
    min_overlap  : minimum interpolated overlap required to estimate a factor.
    """
    # Force luptitude normalisation through flux
    _norm_q = "flux" if norm_quantity == "luptitude" else norm_quantity

    if style is None:
        style = {
            "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),
            "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),
        }

    d = lc_cache[target_oid]

    # ── Working copies ─────────────────────────────────────────────────────────
    frames = [df for df in (d["fp"], d["src"]) if not df.empty]
    df_obj = clean_df(
        pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    )
    df_obj_src = clean_df(d["src"])
    df_obj_fp = clean_df(d["fp"])  # may be empty for new-alert objects

    tmin, tmax = getTminTmax(df_obj, df_obj_src, df_obj_fp)
    bands_obj = [b for b in bands if b in df_obj["r:band"].values]

    # ── Inter-band renormalisation ──────────────────────────────────────────────
    norm_factors = {b: (1.0 if _norm_q == "flux" else 0.0) for b in bands_obj}

    if normalize:
        ref_band, ref_mode = choose_reference_band(df_obj_src, df_obj, min_pts=min_pts)

        if ref_band is None:
            print(f"  ⚠ [{target_oid}] No suitable reference band — skipping normalisation.")
        else:
            print(f"  Reference band: {ref_band} ({ref_mode})")
            df_ref = (
                df_obj_src[df_obj_src["r:band"] == ref_band]
                if ref_mode == "src"
                else df_obj[df_obj["r:band"] == ref_band]
            )

            for band in bands_obj:
                if band == ref_band:
                    continue  # identity

                df_b, _ = select_norm_dataset(df_obj_src, df_obj, band, min_pts=min_pts)
                if df_b is None:
                    print(f"  ⚠ [{target_oid}] Band {band}: not enough data, skipping.")
                    norm_factors[band] = None
                    continue

                factor = compute_band_normalization(df_ref, df_b, quantity=_norm_q, min_overlap=min_overlap)
                norm_factors[band] = factor
                print(f"  Band {band}: factor = {factor}")

            # Apply normalisation on working copies
            for band, factor in norm_factors.items():
                if factor is None:
                    continue
                df_obj = apply_normalization(df_obj, band, factor, _norm_q)
                df_obj_src = apply_normalization(df_obj_src, band, factor, _norm_q)
                # fp may be empty — apply_normalization handles that gracefully
                df_obj_fp = apply_normalization(df_obj_fp, band, factor, _norm_q)

    # ── Layout ─────────────────────────────────────────────────────────────────
    n_rows = (len(bands_obj) if show_per_band else 0) + (1 if show_combined else 0)
    n_cols = len(modes)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.2 * n_rows), squeeze=False)
    mode_to_col = {m: i for i, m in enumerate(modes)}
    current_row = 0

    # ── Core per-panel plot ────────────────────────────────────────────────────
    def plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=False):
        t_src = safe_get_array(df_b_src, "r:midpointMjdTai")
        dt_src = t_src - tmin
        t_fp = safe_get_array(df_b_fp, "r:midpointMjdTai")
        dt_fp = t_fp - tmin

        if mode == "flux":
            mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
            mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
            ax.errorbar(
                dt_src[mask_src],
                safe_get_array(df_b_src, "r:psfFlux")[mask_src],
                yerr=safe_get_array(df_b_src, "r:psfFluxErr")[mask_src],
                color=color,
                **style["src"],
            )
            if mask_fp.sum() > 0:
                ax.errorbar(
                    dt_fp[mask_fp],
                    safe_get_array(df_b_fp, "r:psfFlux")[mask_fp],
                    yerr=safe_get_array(df_b_fp, "r:psfFluxErr")[mask_fp],
                    color=color,
                    **style["fp"],
                )
            ax.axhline(0, color="black", lw=0.6, ls="--", alpha=0.4)
            ax.set_ylabel("Flux (nJy) [renorm]" if normalize else "Flux (nJy)")

        elif mode == "mag":
            mask_src = safe_finite_mask(df_b_src, ["mag", "mag_err"])
            mask_fp = safe_finite_mask(df_b_fp, ["mag", "mag_err"])
            ax.errorbar(
                dt_src[mask_src],
                safe_get_array(df_b_src, "mag")[mask_src],
                yerr=safe_get_array(df_b_src, "mag_err")[mask_src],
                color=color,
                **style["src"],
            )
            if mask_fp.sum() > 0:
                ax.errorbar(
                    dt_fp[mask_fp],
                    safe_get_array(df_b_fp, "mag")[mask_fp],
                    yerr=safe_get_array(df_b_fp, "mag_err")[mask_fp],
                    color=color,
                    **style["fp"],
                )
            ax.invert_yaxis()
            ax.set_ylabel("AB mag [renorm]" if normalize else "AB mag")

        elif mode == "luptitude":
            mask_src = safe_finite_mask(df_b_src, ["r:psfFlux", "r:psfFluxErr"])
            mask_fp = safe_finite_mask(df_b_fp, ["r:psfFlux", "r:psfFluxErr"])
            if mask_src.sum() >= 3:
                lup_src, lup_err_src, _ = flux_to_luptitude(
                    safe_get_array(df_b_src, "r:psfFlux")[mask_src],
                    safe_get_array(df_b_src, "r:psfFluxErr")[mask_src],
                )
                ax.errorbar(dt_src[mask_src], lup_src, yerr=lup_err_src, color=color, **style["src"])
                if mask_fp.sum() > 0:
                    lup_fp, lup_err_fp, _ = flux_to_luptitude(
                        safe_get_array(df_b_fp, "r:psfFlux")[mask_fp],
                        safe_get_array(df_b_fp, "r:psfFluxErr")[mask_fp],
                    )
                    ax.errorbar(dt_fp[mask_fp], lup_fp, yerr=lup_err_fp, color=color, **style["fp"])
                ax.invert_yaxis()
                ax.set_ylabel("Luptitude [renorm]" if normalize else "Luptitude")

        ax.set_xlabel("Δt (days)")
        if not is_combined:
            ax.set_title(f"{mode} — band {band}", fontsize=8, color=color)

    # ── Per-band rows ──────────────────────────────────────────────────────────
    if show_per_band:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = safe_filter_band(df_obj_src, band)
            df_b_fp = safe_filter_band(df_obj_fp, band)
            color = band_colors.get(band, "gray")
            for mode in modes:
                plot_one(
                    axes[current_row][mode_to_col[mode]],
                    df_b,
                    df_b_src,
                    df_b_fp,
                    mode,
                    band,
                    color,
                )
            current_row += 1

    # ── Combined row ───────────────────────────────────────────────────────────
    if show_combined:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = safe_filter_band(df_obj_src, band)
            df_b_fp = safe_filter_band(df_obj_fp, band)
            color = band_colors.get(band, "gray")
            for mode in modes:
                plot_one(
                    axes[current_row][mode_to_col[mode]],
                    df_b,
                    df_b_src,
                    df_b_fp,
                    mode,
                    band,
                    color,
                    is_combined=True,
                )
        for mode in modes:
            ax = axes[current_row][mode_to_col[mode]]
            ax.set_title(f"{mode} (all bands) [renorm]" if normalize else f"{mode} (all bands)")
            if mode in ("mag", "luptitude"):
                ax.invert_yaxis()

    # ── Final touches ──────────────────────────────────────────────────────────
    norm_label = f"renorm={norm_quantity}" if normalize else "no renorm"
    fig.suptitle(
        f"diaObjectId = {target_oid} | group = {target_group} | {norm_label}",
        fontsize=11,
    )
    plt.tight_layout()
    if save:
        safe = target_group.replace("/", "_").replace(" ", "_")
        plt.savefig(
            os.path.join(DIR_FIGS, f"{filename_prefix}_{safe}_{target_oid}.png"),
            bbox_inches="tight",
        )
    plt.show()


print("plot_singleobjlightcurve_normalized() defined.")

## 13. Renormalised view — single target object

In [ ]:
print(f">>> diaObjectId = {TARGET_OID} | group = {TARGET_TAG} <<<")
plot_singleobjlightcurve_normalized(
    lc_cache,
    TARGET_TAG,
    TARGET_OID,
    BANDS,
    BAND_COLORS,
    modes=("flux", "mag", "luptitude"),
    show_per_band=False,
    show_combined=True,
    normalize=True,
    norm_quantity="flux",
)

## 14. Renormalised view — all tags, all objects (combined panel only)

Loops over all selected tags and all top-NC objects.
Each object is displayed in a single row (combined panel): flux / mag / luptitude.

In [ ]:
groups_to_plot = [g for g in selected_tags if g in group_oids and len(group_oids[g]) >= 1]

for target_group in groups_to_plot:
    n_obj = len(group_oids[target_group])
    desc = FINK_TAGS.get(target_group, "")
    print(f"\n{'=' * 70}")
    print(f"=== TAG: {target_group}  ({n_obj} objects)")
    if desc:
        print(f"    {desc}")
    print(f"{'=' * 70}")

    obj_numbers = rank_oids(group_oids[target_group], nc=NC)

    for target_oid in obj_numbers:
        print(f">>> diaObjectId = {target_oid} | group = {target_group}")
        plot_singleobjlightcurve_normalized(
            lc_cache,
            target_group,
            target_oid,
            BANDS,
            BAND_COLORS,
            modes=("flux", "mag", "luptitude"),
            show_per_band=False,
            show_combined=True,
            normalize=True,
            norm_quantity="flux",
            save=True,
            filename_prefix="05_renorm",
        )

## 15. Summary scatter plot — variability vs mean flux per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"].replace("_", "\n"),
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Fink tag variability summary (objects in DDFs)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("06_tag_summary_scatter")
    plt.show()

    print("\nTop rows sorted by (band, median_rms):")
    display(summary.sort_values(["band", "median_rms"]).head(40))

## 16. Final ranking table — variability by tag

In [ ]:
if not df_ranking.empty:
    print("Ranking by photometric variability (ascending RMS) — all Fink tags in DDFs:")
    print("=" * 110)
    cols = [
        c
        for c in ["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]
        if c in df_ranking.columns
    ]
    print(df_ranking[cols].to_string(index=False, float_format="{:.4f}".format))

    if not df_flat.empty and "field" in df_flat.columns:
        print("\nMedian RMS by (tag, DDF):")
        breakdown = (
            df_flat.groupby(["group", "field"])
            .agg(n_obj=("diaObjectId", "nunique"), median_rms=("rms_var", "median"))
            .reset_index()
            .sort_values(["group", "median_rms"])
        )
        display(breakdown)

elif not df_flat.empty:
    print("tag_ranking.csv not found — recomputing from flatness_metrics.csv.")
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    ranking["description"] = ranking["group"].map(FINK_TAGS).fillna("")
    display(ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]])
else:
    print("No ranking data available.")